#### Simple Gen AI APP Using LangChain and LangGraph

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")
# LangSmith Tracking
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANG_SMITH_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

In [2]:
## Data Ingestion--From the website we need to scrape the data
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://docs.smith.langchain.com/tutorials/Administrators/manage_spend")
loader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
docs = loader.load()
docs
# docs[0].page_content[:500]

[Document(metadata={'source': 'https://docs.smith.langchain.com/tutorials/Administrators/manage_spend', 'title': 'LangSmith Observability - Docs by LangChain', 'description': 'Instrument your LLM application, investigate traces, and monitor performance in production with LangSmith.', 'language': 'en'}, page_content="LangSmith Observability - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewTraceDebugObserveReferenceLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance metrics.LangSmith works wi

In [4]:
### Load Data --> Docs --> Divide our text into chunks --> text --> vectors --> Vector Embeddings --> Vector Store
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(docs)

In [5]:
documents

[Document(metadata={'source': 'https://docs.smith.langchain.com/tutorials/Administrators/manage_spend', 'title': 'LangSmith Observability - Docs by LangChain', 'description': 'Instrument your LLM application, investigate traces, and monitor performance in production with LangSmith.', 'language': 'en'}, page_content="LangSmith Observability - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewTraceDebugObserveReferenceLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance metrics.LangSmith works wi

In [6]:

from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=os.environ["GEMINI_API_KEY"]
)



In [7]:
from langchain_community.vectorstores import FAISS

vectorstoredb = FAISS.from_documents(documents, embeddings)

In [8]:
vectorstoredb

In [9]:
query= "LangSmith works with many frameworks and providers."
results = vectorstoredb.similarity_search(query)
results[0].page_content

"LangSmith Observability - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewTraceDebugObserveReferenceLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance metrics.LangSmith works with many frameworks and providers. Browse available integrations to connect your stack including OpenAI, Anthropic, CrewAI, Vercel AI SDK, Pydantic AI, and more.Get startedCreate an accountSign up at smith.langchain.com (no credit card required)."

In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash",google_api_key=os.environ["GEMINI_API_KEY"])

In [11]:
# Document Chain

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context:

<context>
{context}
</context>

Question:
{input}
"""
)

document_chain = (
    prompt
    | llm
    | StrOutputParser()
)

document_chain

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n\n<context>\n{context}\n</context>\n\nQuestion:\n{input}\n'), additional_kwargs={})])
| ChatGoogleGenerativeAI(model='models/gemini-2.5-flash', google_api_key=SecretStr('**********'), client=<google.ai.generativelanguage_v1beta.services.generative_service.client.GenerativeServiceClient object at 0x000002A431F94A50>, default_metadata=(), model_kwargs={})
| StrOutputParser()

In [12]:
response = document_chain.invoke({
    "context": "LangSmith Observability LangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance metrics.LangSmith works with many frameworks and providers. Browse available integrations to connect your stack including OpenAI, Anthropic, CrewAI, Vercel AI SDK, Pydantic AI, and more.",
    "input": "What is LangSmith?"
})

print(response)

LangSmith is an Observability tool that provides full visibility into LLM applications, from individual traces to production-wide performance metrics. It works with many frameworks and providers.


In [14]:
# Retriever

retriever=vectorstoredb.as_retriever()

from langchain.chains import create_retrieval_chain

retrieval_chain = create_retrieval_chain(retriever, document_chain)

In [15]:
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002A431E5B620>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n\n<context>\n{context}\n</context>\n\nQuestion:\n{input}\n'), additional_kwargs={})])
            | ChatGoogleGenerativeAI(model='models/gemini-2.5-flash', google_api_key=SecretStr('**********'), client=<google.ai.generativelanguage_v1beta.services.generative_service.c

In [16]:
# Get the response from llm
response = retrieval_chain.invoke({"input": "What is LangSmith?"})
response['answer']

'LangSmith is a tool that provides full visibility into LLM applications, allowing users to instrument their LLM applications, investigate traces, and monitor performance in production. It offers capabilities ranging from individual traces to production-wide performance metrics.\n\nKey features and functionalities of LangSmith include:\n*   **Observability**\n*   **Evaluation**\n*   **Prompt engineering**\n*   **Deployment**\n*   **Trace investigation**: Users can view, filter, export, share, and compare traces via the UI or API.\n*   **Performance monitoring**: It allows building dashboards and setting alerts to track quality and catch issues early.\n*   **Automation**: Users can configure workflows with rules, webhooks, and online evaluations.\n*   **Feedback collection**: It enables annotating outputs and gathering user feedback using queues or inline annotation.\n*   **LangSmith Engine**: This component automatically detects recurring issues in traces, diagnoses their root cause, a

In [17]:
print(response.keys())

dict_keys(['input', 'context', 'answer'])


In [18]:
response

{'input': 'What is LangSmith?',
 'context': [Document(id='3b9551df-7db3-4996-862e-3c145b570f03', metadata={'source': 'https://docs.smith.langchain.com/tutorials/Administrators/manage_spend', 'title': 'LangSmith Observability - Docs by LangChain', 'description': 'Instrument your LLM application, investigate traces, and monitor performance in production with LangSmith.', 'language': 'en'}, page_content='cause, and resolve them with LangSmith Engine.For terminology and core concepts, refer to Observability concepts. For trace pricing, retention, and limits, see Usage and billing.To set up a LangSmith instance, visit the Platform setup section to choose between cloud, hybrid, or self-hosted. All options include observability, evaluation, prompt engineering, and deployment.'),
  Document(id='a62e951e-c07c-49df-8407-61549888646c', metadata={'source': 'https://docs.smith.langchain.com/tutorials/Administrators/manage_spend', 'title': 'LangSmith Observability - Docs by LangChain', 'description'